# Каскад YOLOv8s + ResNet-18 для распознавания сигналов светофора

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import deque
import json
import time
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torchvision import models, transforms
from ultralytics import YOLO
import torch.nn as nn

@dataclass
class CascadeConfig:
    ROOT_DIR: str = "."
    VIDEOS_DIR: str = "Videos"
    OUT_DIR: str = "runs_cascade_yolov8s"

    YOLO_WEIGHTS: str = "yolov8s.pt"
    CLS_WEIGHTS: str = "tl_custom_train/resnet18_tl_state_dict.pt"
    CLS_META: str = "tl_custom_train/resnet18_tl_custom_meta.json"

    USE_CLASSIFIER: bool = True
    TL_CLASS_ID: int = 9

    # Детекция и геометрия
    MIN_CONF: float = 0.35
    TARGET_X_RATIO: float = 0.50
    TARGET_Y_RATIO: float = 0.19
    X_RANGE: tuple = (0.10, 0.94)
    Y_RANGE: tuple = (0.02, 0.48)

    MIN_AREA: int = 36
    MAX_AREA: int = 14000
    MIN_VERT_RATIO: float = 1.05
    MAX_VERT_RATIO: float = 4.30
    CROP_PAD_RATIO: float = 0.22
    MIN_SHARPNESS: float = 10.0

    # Пороги классификатора
    CLS_CONF_THR: float = 0.75
    CANDIDATE_CLS_THR: float = 0.72

    # Логика выбора / сопровождения
    SEARCH_SCORE_THR: float = 1.10
    REACQUIRE_SCORE_THR: float = 1.00
    TRACK_SCORE_THR: float = 2.40
    HOLD_FRAMES: int = 2
    SMOOTH_WINDOW: int = 3
    GARBAGE_PATIENCE: int = 2

    # Best frame
    BEST_FRAME_SKIP_FRAMES: int = 60
    BEST_FRAME_MIN_AREA: int = 220
    BEST_FRAME_MIN_STABLE: int = 5
    BEST_FRAME_MIN_CLS: float = 0.90
    BEST_FRAME_SHARP_CAP: float = 4000.0

    # Сохранение
    SAVE_VIDEO: bool = True
    SAVE_CROPS: bool = True
    SAVE_CROP_EVERY: int = 1
    DRAW_ALL_CANDIDATES: bool = True

    # Производительность
    PROCESS_EVERY_N: int = 1

    # Фильтр по имени видео
    VIDEO_FILTER: str | None = None

cfg = CascadeConfig()
ROOT = Path(cfg.ROOT_DIR)
VIDEOS_DIR = ROOT / cfg.VIDEOS_DIR
OUT_ROOT = ROOT / cfg.OUT_DIR
OUT_ROOT.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(cfg)
print("device:", device)

In [ ]:

def find_videos(videos_dir: Path, video_filter=None):
    exts = {".mp4", ".mov", ".avi", ".mkv", ".MP4", ".MOV", ".AVI", ".MKV"}
    files = [p for p in videos_dir.iterdir() if p.suffix in exts and p.is_file()]
    files = sorted(files)
    if video_filter:
        if isinstance(video_filter, str):
            files = [p for p in files if video_filter.lower() in p.name.lower()]
        elif isinstance(video_filter, (list, tuple, set)):
            names = {str(x).lower() for x in video_filter}
            files = [p for p in files if p.name.lower() in names]
    return files
video_files = find_videos(VIDEOS_DIR, cfg.VIDEO_FILTER)
print("Найдено видео:", len(video_files))
for p in video_files:
    print(" -", p.name)


In [ ]:

def load_classifier_meta(meta_path: str):
    meta_file = ROOT / meta_path
    if meta_file.exists():
        with open(meta_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
        print("meta.json загружен:", meta_file)
        return meta

    print("meta.json не найден, будет использован дефолт: 0=red, 1=green")
    return {
        "class_to_idx": {"red": 0, "green": 1},
        "idx_to_class": {"0": "red", "1": "green"},
        "input_size": 96,
        "imagenet_mean": [0.485, 0.456, 0.406],
        "imagenet_std": [0.229, 0.224, 0.225],
        "notes": "fallback meta: 0=red, 1=green",
    }

cls_meta = load_classifier_meta(cfg.CLS_META)
CLASS_TO_IDX = {str(k): int(v) for k, v in cls_meta.get("class_to_idx", {"red": 0, "green": 1}).items()}
IDX_TO_CLASS_RAW = cls_meta.get("idx_to_class", {"0": "red", "1": "green"})
IDX_TO_CLASS = {int(k): str(v) for k, v in IDX_TO_CLASS_RAW.items()}
INPUT_SIZE = int(cls_meta.get("input_size", 96))
IMAGENET_MEAN = tuple(cls_meta.get("imagenet_mean", [0.485, 0.456, 0.406]))
IMAGENET_STD = tuple(cls_meta.get("imagenet_std", [0.229, 0.224, 0.225]))
RED_IDX = CLASS_TO_IDX.get("red", 0)
GREEN_IDX = CLASS_TO_IDX.get("green", 1)
print("CLASS_TO_IDX:", CLASS_TO_IDX)
print("RED_IDX:", RED_IDX, "| GREEN_IDX:", GREEN_IDX)
print("INPUT_SIZE:", INPUT_SIZE)
cls_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def build_classifier():
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)

def load_classifier(weights_path: str):
    weights_file = ROOT / weights_path
    if not weights_file.exists():
        raise FileNotFoundError(f"Не найдены веса классификатора: {weights_file}")

    model = build_classifier()
    state = torch.load(weights_file, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Загружены веса классификатора:", weights_file)
    return model

cls_model = load_classifier(cfg.CLS_WEIGHTS) if cfg.USE_CLASSIFIER else None
print("Загрузка YOLO:", cfg.YOLO_WEIGHTS)
yolo_model = YOLO(cfg.YOLO_WEIGHTS)


In [ ]:

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(1.0, ax2 - ax1) * max(1.0, ay2 - ay1)
    area_b = max(1.0, bx2 - bx1) * max(1.0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-6)

def bbox_center(box):
    x1, y1, x2, y2 = box
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))

def bbox_area(box):
    x1, y1, x2, y2 = box
    return max(1.0, x2 - x1) * max(1.0, y2 - y1)

def expand_box(box, width, height, pad_ratio):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1
    px = int(round(bw * pad_ratio))
    py = int(round(bh * pad_ratio))
    nx1 = max(0, int(round(x1 - px)))
    ny1 = max(0, int(round(y1 - py)))
    nx2 = min(width, int(round(x2 + px)))
    ny2 = min(height, int(round(y2 + py)))
    return nx1, ny1, nx2, ny2

def laplacian_sharpness(image_bgr):
    if image_bgr is None or image_bgr.size == 0:
        return 0.0
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def largest_cc_ratio(mask):
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return 0.0
    largest = stats[1:, cv2.CC_STAT_AREA].max()
    return float(largest) / float(mask.shape[0] * mask.shape[1] + 1e-6)

def mask_density(mask):
    return float(mask.mean()) / 255.0

def signal_color_score(roi_bgr):
    if roi_bgr is None or roi_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    red1 = cv2.inRange(hsv, (0, 80, 110), (15, 255, 255))
    red2 = cv2.inRange(hsv, (160, 80, 110), (180, 255, 255))
    red = cv2.bitwise_or(red1, red2)
    yellow = cv2.inRange(hsv, (15, 65, 120), (40, 255, 255))
    green = cv2.inRange(hsv, (35, 60, 85), (95, 255, 255))
    scores = []
    for mask in (red, yellow, green):
        dens = mask_density(mask)
        cc = largest_cc_ratio(mask)
        score = 4.0 * cc + 2.5 * dens
        scores.append(score)
    return float(max(scores)) if scores else 0.0

def classify_roi(roi_bgr, cls_model, cls_tf, device):
    roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)
    x = cls_tf(roi_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = cls_model(x)
        probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    p_red = float(probs[RED_IDX])
    p_green = float(probs[GREEN_IDX])
    if p_red >= p_green:
        pred_label = "red"
        pred_conf = p_red
    else:
        pred_label = "green"
        pred_conf = p_green
    return {
        "p_red": p_red,
        "p_green": p_green,
        "pred_label": pred_label,
        "pred_conf": pred_conf,
        "margin": abs(p_red - p_green),
    }

def geometry_score(box, conf, frame_w, frame_h, cfg):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 0 or bh <= 0:
        return None
    area = bw * bh
    cx, cy = bbox_center(box)
    vert_ratio = bh / max(bw, 1.0)
    if conf < cfg.MIN_CONF:
        return None
    if not (cfg.X_RANGE[0] * frame_w <= cx <= cfg.X_RANGE[1] * frame_w):
        return None
    if not (cfg.Y_RANGE[0] * frame_h <= cy <= cfg.Y_RANGE[1] * frame_h):
        return None
    if not (cfg.MIN_AREA <= area <= cfg.MAX_AREA):
        return None
    if not (cfg.MIN_VERT_RATIO <= vert_ratio <= cfg.MAX_VERT_RATIO):
        return None
    tx = cfg.TARGET_X_RATIO * frame_w
    ty = cfg.TARGET_Y_RATIO * frame_h
    dist = abs(cx - tx) / frame_w + abs(cy - ty) / frame_h
    area_score = min(area / 2200.0, 1.0)
    return 2.1 * conf + 0.7 * area_score - 2.5 * dist


In [ ]:

def extract_candidates(result, frame, cfg, cls_model, cls_tf, device):
    frame_h, frame_w = frame.shape[:2]
    candidates = []
    for b in result.boxes:
        cls_id = int(b.cls[0].item())
        if cls_id != cfg.TL_CLASS_ID:
            continue
        conf = float(b.conf[0].item())
        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
        box = (x1, y1, x2, y2)
        geom = geometry_score(box, conf, frame_w, frame_h, cfg)
        if geom is None:
            continue
        ex1, ey1, ex2, ey2 = expand_box(box, frame_w, frame_h, cfg.CROP_PAD_RATIO)
        roi = frame[ey1:ey2, ex1:ex2].copy()
        sharpness = laplacian_sharpness(roi)
        color_score = signal_color_score(roi)
        cls_info = {
            "p_red": None,
            "p_green": None,
            "pred_label": "unknown",
            "pred_conf": 0.0,
            "margin": 0.0,
        }
        cls_bonus = 0.0
        if (
            cfg.USE_CLASSIFIER
            and cls_model is not None
            and roi.size > 0
            and sharpness >= cfg.MIN_SHARPNESS
        ):
            cls_info = classify_roi(roi, cls_model, cls_tf, device)
            if cls_info["pred_conf"] >= cfg.CANDIDATE_CLS_THR:
                cls_bonus = 1.35 * cls_info["pred_conf"] + 0.90 * cls_info["margin"]
            else:
                cls_bonus = -0.60
        combined_score = float(geom + 1.25 * color_score + cls_bonus)
        candidates.append({
            "bbox": box,
            "conf": conf,
            "geom_score": float(geom),
            "combined_score": combined_score,
            "color_score": float(color_score),
            "sharpness": float(sharpness),
            "cls_label": cls_info["pred_label"],
            "cls_best": float(cls_info["pred_conf"]),
            "cls_margin": float(cls_info["margin"]),
            "p_red": cls_info["p_red"],
            "p_green": cls_info["p_green"],
            "cx": float((x1 + x2) / 2.0),
            "cy": float((y1 + y2) / 2.0),
            "area": float((x2 - x1) * (y2 - y1)),
        })
    candidates.sort(key=lambda x: x["combined_score"], reverse=True)
    return candidates

def match_to_previous(prev_bbox, candidates, frame_w, frame_h):
    if prev_bbox is None or not candidates:
        return None
    prev_area = bbox_area(prev_bbox)
    prev_cx, prev_cy = bbox_center(prev_bbox)
    best = None
    best_score = -1e9
    for cand in candidates:
        bb = cand["bbox"]
        iou = iou_xyxy(prev_bbox, bb)
        cx, cy = bbox_center(bb)
        dist = abs(cx - prev_cx) / frame_w + abs(cy - prev_cy) / frame_h
        area = bbox_area(bb)
        area_ratio = min(prev_area, area) / max(prev_area, area)
        track_score = (
            1.15 * cand["combined_score"]
            + 2.70 * iou
            + 0.35 * cand["conf"]
            + 0.45 * area_ratio
            - 1.35 * dist
        )
        item = cand.copy()
        item["iou_prev"] = float(iou)
        item["track_score"] = float(track_score)
        if track_score > best_score:
            best_score = track_score
            best = item
    return best

def draw_box(frame, box, color, text=None, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    if text:
        cv2.putText(
            frame,
            text,
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
            cv2.LINE_AA,
        )


In [ ]:
def process_video(video_path, cfg, yolo_model, cls_model, cls_tf, device):
    video_path = Path(video_path)
    stem = video_path.stem
    out_dir = OUT_ROOT / stem
    crops_dir = out_dir / "crops"
    out_dir.mkdir(parents=True, exist_ok=True)
    if cfg.SAVE_CROPS:
        crops_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    writer = None
    if cfg.SAVE_VIDEO:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(
            str(out_dir / f"{stem}_annotated.mp4"),
            fourcc,
            fps,
            (frame_w, frame_h),
        )
    prev_bbox = None
    hold_bbox = None
    hold_left = 0
    lost_frames = 0
    garbage_frames = 0
    tracker_state = "SEARCH"
    prob_history = deque(maxlen=cfg.SMOOTH_WINDOW)
    logs = []
    best_frame_score = -1e9
    best_frame_bgr = None
    best_crop_bgr = None
    best_bbox = None
    stable_locked_count = 0
    start_ts = time.perf_counter()
    frame_idx = 0
    pbar = tqdm(total=total_frames if total_frames > 0 else None, desc=stem)
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_t0 = time.perf_counter()
        if frame_idx % cfg.PROCESS_EVERY_N != 0:
            if writer is not None:
                writer.write(frame)
            latency_ms = (time.perf_counter() - frame_t0) * 1000.0
            logs.append({
                "video": stem,
                "frame_idx": int(frame_idx),
                "tracker_state": "SKIPPED",
                "mode": "skipped",
                "has_bbox": 0,
                "bbox_x1": None,
                "bbox_y1": None,
                "bbox_x2": None,
                "bbox_y2": None,
                "det_conf": None,
                "geom_score": None,
                "combined_score": None,
                "color_score": None,
                "track_score": None,
                "iou_prev": None,
                "sharpness": None,
                "cand_cls_best": None,
                "cand_cls_label": None,
                "cand_p_red": None,
                "cand_p_green": None,
                "p_red_raw": None,
                "p_green_raw": None,
                "label_raw": "unknown",
                "p_red_smooth": None,
                "p_green_smooth": None,
                "label_smooth": "unknown",
                "lost_frames": int(lost_frames),
                "crop_path": None,
                "latency_ms": float(latency_ms),
            })
            frame_idx += 1
            pbar.update(1)
            continue
        result = yolo_model(frame, verbose=False, conf=cfg.MIN_CONF)[0]
        candidates = extract_candidates(result, frame, cfg, cls_model, cls_tf, device)
        search_best = candidates[0] if candidates else None
        locked_best = match_to_previous(prev_bbox, candidates, frame_w, frame_h)
        selected = None
        active_bbox = None
        active_mode = "none"
        det_conf = None
        geom_score = None
        combined_score = None
        color_score = None
        track_score = None
        iou_prev = None
        cand_cls_best = None
        cand_cls_label = None
        cand_p_red = None
        cand_p_green = None
        if prev_bbox is None:
            if search_best is not None and search_best["combined_score"] >= cfg.SEARCH_SCORE_THR:
                selected = search_best
                active_mode = "search_acquire"
                tracker_state = "LOCKED"
        else:
            if locked_best is not None and locked_best["track_score"] >= cfg.TRACK_SCORE_THR:
                selected = locked_best
                active_mode = "locked"
                tracker_state = "LOCKED"
            elif search_best is not None and search_best["combined_score"] >= cfg.REACQUIRE_SCORE_THR:
                selected = search_best
                active_mode = "reacquire"
                tracker_state = "REACQUIRE"
            elif hold_bbox is not None and hold_left > 0:
                active_bbox = hold_bbox
                hold_left -= 1
                lost_frames += 1
                active_mode = "hold"
                tracker_state = "LOST"
            else:
                prev_bbox = None
                hold_bbox = None
                hold_left = 0
                lost_frames += 1
                tracker_state = "SEARCH"

        if selected is not None:
            active_bbox = selected["bbox"]
            prev_bbox = active_bbox
            hold_bbox = active_bbox
            hold_left = cfg.HOLD_FRAMES
            lost_frames = 0

            det_conf = float(selected["conf"])
            geom_score = float(selected["geom_score"])
            combined_score = float(selected["combined_score"])
            color_score = float(selected["color_score"])
            track_score = float(selected["track_score"]) if "track_score" in selected else None
            iou_prev = float(selected["iou_prev"]) if "iou_prev" in selected else None
            cand_cls_best = float(selected["cls_best"])
            cand_cls_label = selected["cls_label"]
            cand_p_red = selected["p_red"]
            cand_p_green = selected["p_green"]

        sharpness = None
        p_red = None
        p_green = None
        raw_label = "unknown"
        smooth_label = "unknown"
        smooth_red = None
        smooth_green = None
        crop_path = None
        roi = None
        if active_bbox is not None:
            ex1, ey1, ex2, ey2 = expand_box(active_bbox, frame_w, frame_h, cfg.CROP_PAD_RATIO)
            roi = frame[ey1:ey2, ex1:ex2].copy()
            if roi.size > 0:
                sharpness = laplacian_sharpness(roi)
                if (
                    cfg.USE_CLASSIFIER
                    and cls_model is not None
                    and sharpness >= cfg.MIN_SHARPNESS
                ):
                    cls_info = classify_roi(roi, cls_model, cls_tf, device)
                    p_red = float(cls_info["p_red"])
                    p_green = float(cls_info["p_green"])
                    if cls_info["pred_conf"] < cfg.CLS_CONF_THR:
                        raw_label = "garbage"
                        garbage_frames += 1
                        prob_history.clear()
                    else:
                        raw_label = cls_info["pred_label"]
                        garbage_frames = 0
                        prob_history.append(np.array([p_red, p_green], dtype=np.float32))
                else:
                    raw_label = "unknown"
                    garbage_frames = 0
                if cfg.SAVE_CROPS and frame_idx % cfg.SAVE_CROP_EVERY == 0:
                    crop_path = crops_dir / f"{stem}_f{frame_idx:06d}.png"
                    cv2.imwrite(str(crop_path), roi)
            else:
                prob_history.clear()
                garbage_frames = 0
        else:
            prob_history.clear()
            garbage_frames = 0
        if garbage_frames >= cfg.GARBAGE_PATIENCE:
            prev_bbox = None
            hold_bbox = None
            hold_left = 0
            active_bbox = None
            active_mode = "rejected"
            tracker_state = "SEARCH"
            det_conf = None
            geom_score = None
            combined_score = None
            color_score = None
            track_score = None
            iou_prev = None
            sharpness = None
            raw_label = "garbage"
            prob_history.clear()
            garbage_frames = 0
            roi = None

        if len(prob_history) > 0:
            smooth_probs = np.mean(np.stack(prob_history, axis=0), axis=0)
            smooth_red = float(smooth_probs[0])
            smooth_green = float(smooth_probs[1])
            smooth_label = "red" if smooth_red >= smooth_green else "green"
        else:
            smooth_label = "garbage" if raw_label == "garbage" else raw_label

        if (
            active_bbox is not None
            and tracker_state == "LOCKED"
            and active_mode == "locked"
            and lost_frames == 0
            and raw_label in {"red", "green"}
            and smooth_label == raw_label
        ):
            stable_locked_count += 1
        else:
            stable_locked_count = 0

        if roi is not None and roi.size > 0 and active_bbox is not None:
            bx1, by1, bx2, by2 = active_bbox
            area = float((bx2 - bx1) * (by2 - by1))
            cx = 0.5 * (bx1 + bx2)
            cy = 0.5 * (by1 + by2)

            tx = cfg.TARGET_X_RATIO * frame_w
            ty = cfg.TARGET_Y_RATIO * frame_h
            center_penalty = abs(cx - tx) / frame_w + abs(cy - ty) / frame_h

            area_score = min(area / 900.0, 1.0)
            sharp_score = min((sharpness or 0.0) / cfg.BEST_FRAME_SHARP_CAP, 1.0)
            det_score = float(det_conf or 0.0)
            cls_score = float(cand_cls_best or 0.0)
            iou_score = float(max(0.0, min(1.0, iou_prev if iou_prev is not None else 0.0)))
            stable_score = min(stable_locked_count / float(cfg.BEST_FRAME_MIN_STABLE + 3), 1.0)

            is_good_best_frame = (
                frame_idx >= cfg.BEST_FRAME_SKIP_FRAMES
                and tracker_state == "LOCKED"
                and active_mode == "locked"
                and lost_frames == 0
                and raw_label in {"red", "green"}
                and smooth_label == raw_label
                and cls_score >= cfg.BEST_FRAME_MIN_CLS
                and area >= cfg.BEST_FRAME_MIN_AREA
                and stable_locked_count >= cfg.BEST_FRAME_MIN_STABLE
            )

            if is_good_best_frame:
                rank_score = (
                    2.0 * stable_score
                    + 1.4 * cls_score
                    + 1.1 * det_score
                    + 1.0 * area_score
                    + 0.8 * iou_score
                    + 0.35 * sharp_score
                    - 1.7 * center_penalty
                )

                if rank_score > best_frame_score:
                    best_frame_score = rank_score
                    best_frame_bgr = frame.copy()
                    best_crop_bgr = roi.copy()
                    best_bbox = active_bbox

        vis = frame.copy()

        if cfg.DRAW_ALL_CANDIDATES:
            for cand in candidates:
                cand_text = (
                    f"c={cand['conf']:.2f} "
                    f"g={cand['geom_score']:.2f} "
                    f"m={cand['combined_score']:.2f}"
                )
                draw_box(vis, cand["bbox"], (0, 255, 255), text=cand_text, thickness=1)

        if active_bbox is not None:
            if smooth_label == "green":
                color = (0, 255, 0)
            elif smooth_label == "red":
                color = (0, 0, 255)
            elif smooth_label == "garbage":
                color = (255, 200, 0)
            else:
                color = (255, 255, 0)

            if active_mode == "hold":
                color = (0, 165, 255)

            line1 = f"{active_mode} | {tracker_state}"
            line2 = (
                f"det={det_conf:.2f} geom={geom_score:.2f} comb={combined_score:.2f}"
                if det_conf is not None and geom_score is not None and combined_score is not None
                else "det=None"
            )

            if smooth_red is not None and smooth_green is not None:
                line3 = f"raw={raw_label} | smooth={smooth_label} ({smooth_red:.2f}/{smooth_green:.2f})"
            else:
                line3 = f"raw={raw_label} | smooth={smooth_label}"

            line4 = (
                f"sharp={sharpness:.1f} color={color_score:.2f} cand={cand_cls_label}/{cand_cls_best:.2f}"
                if sharpness is not None and color_score is not None and cand_cls_best is not None and cand_cls_label is not None
                else f"sharp={sharpness:.1f}" if sharpness is not None else "sharp=None"
            )

            draw_box(vis, active_bbox, color, text=line1, thickness=3)
            cv2.putText(vis, line2, (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
            cv2.putText(vis, line3, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
            cv2.putText(vis, line4, (20, 88), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
        else:
            cv2.putText(
                vis,
                f"state={tracker_state}",
                (20, 32),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (220, 220, 220),
                2,
                cv2.LINE_AA,
            )

        cv2.putText(
            vis,
            f"{stem} | frame={frame_idx}",
            (20, frame_h - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        if writer is not None:
            writer.write(vis)

        latency_ms = (time.perf_counter() - frame_t0) * 1000.0

        logs.append({
            "video": stem,
            "frame_idx": int(frame_idx),
            "tracker_state": tracker_state,
            "mode": active_mode,
            "has_bbox": int(active_bbox is not None),
            "bbox_x1": float(active_bbox[0]) if active_bbox is not None else None,
            "bbox_y1": float(active_bbox[1]) if active_bbox is not None else None,
            "bbox_x2": float(active_bbox[2]) if active_bbox is not None else None,
            "bbox_y2": float(active_bbox[3]) if active_bbox is not None else None,
            "det_conf": det_conf,
            "geom_score": geom_score,
            "combined_score": combined_score,
            "color_score": color_score,
            "track_score": track_score,
            "iou_prev": iou_prev,
            "sharpness": sharpness,
            "cand_cls_best": cand_cls_best,
            "cand_cls_label": cand_cls_label,
            "cand_p_red": cand_p_red,
            "cand_p_green": cand_p_green,
            "p_red_raw": p_red,
            "p_green_raw": p_green,
            "label_raw": raw_label,
            "p_red_smooth": smooth_red,
            "p_green_smooth": smooth_green,
            "label_smooth": smooth_label,
            "lost_frames": int(lost_frames),
            "crop_path": str(crop_path) if crop_path is not None else None,
            "latency_ms": float(latency_ms),
        })

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    if writer is not None:
        writer.release()

    elapsed = time.perf_counter() - start_ts
    ms_per_frame = (elapsed / max(frame_idx, 1)) * 1000.0

    df = pd.DataFrame(logs)
    df.to_csv(out_dir / f"{stem}_frame_log.csv", index=False, encoding="utf-8-sig")

    with open(out_dir / f"{stem}_track_log.json", "w", encoding="utf-8") as f:
        json.dump({
            "video": str(video_path),
            "frames_processed": int(frame_idx),
            "fps": fps,
            "width": frame_w,
            "height": frame_h,
            "elapsed_sec": elapsed,
            "ms_per_frame_total": ms_per_frame,
            "config": asdict(cfg),
            "classifier_meta": cls_meta,
            "logs": logs,
        }, f, ensure_ascii=False, indent=2)

    if best_frame_bgr is not None:
        if best_bbox is not None:
            draw_box(best_frame_bgr, best_bbox, (255, 255, 255), text="best frame", thickness=3)
        cv2.imwrite(str(out_dir / f"{stem}_best_frame.png"), best_frame_bgr)

    if best_crop_bgr is not None:
        cv2.imwrite(str(out_dir / f"{stem}_best_crop.png"), best_crop_bgr)

    summary = {
        "video": stem,
        "frames": int(frame_idx),
        "fps": fps,
        "ms_per_frame_total": float(ms_per_frame),
        "mean_latency_ms": float(df["latency_ms"].mean()) if len(df) else None,
        "max_latency_ms": float(df["latency_ms"].max()) if len(df) else None,
        "bbox_frames": int(df["has_bbox"].sum()) if len(df) else 0,
        "bbox_ratio": float(df["has_bbox"].mean()) if len(df) else 0.0,
        "mean_det_conf": float(df["det_conf"].dropna().mean()) if df["det_conf"].notna().any() else None,
        "mean_sharpness": float(df["sharpness"].dropna().mean()) if df["sharpness"].notna().any() else None,
        "red_frames": int((df["label_smooth"] == "red").sum()) if len(df) else 0,
        "green_frames": int((df["label_smooth"] == "green").sum()) if len(df) else 0,
        "garbage_frames": int((df["label_smooth"] == "garbage").sum()) if len(df) else 0,
        "last_smooth_label": None if len(df) == 0 else df["label_smooth"].iloc[-1],
        "out_dir": str(out_dir),
    }

    return summary

In [ ]:

summaries = []
if len(video_files) == 0:
    print("Видео для обработки не найдены. Проверь cfg.VIDEOS_DIR и cfg.VIDEO_FILTER.")
else:
    for video_path in video_files:
        print(f"\n=== Обработка: {video_path.name} ===")
        summary = process_video(video_path, cfg, yolo_model, cls_model, cls_tf, device)
        summaries.append(summary)
summary_df = pd.DataFrame(summaries)
summary_df


In [ ]:

if len(summary_df):
    summary_path = OUT_ROOT / "summary.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print("Сводка сохранена:", summary_path)
    display(summary_df)
